<a href="https://colab.research.google.com/github/18felasofa/Analisis-Deret-Waktu/blob/main/adw_parameter_estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline Box–Jenkins dengan Estimasi Parameter

Notebook ini merupakan versi `.ipynb` dari pipeline `adw_parameter_estimation.py`. Tahap estimasi parameter model terbaik disertakan sehingga nilai parameter AR, MA, konstanta, standard error, z-value, p-value, dan confidence interval dapat ditampilkan.

In [1]:
import os
import seaborn as sns
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, boxcox, shapiro
from scipy.special import inv_boxcox
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.graphics.gofplots import qqplot

warnings.simplefilter('ignore', ConvergenceWarning)
warnings.filterwarnings("ignore")

In [2]:
# =====================================================================

In [3]:
# 1. HELPER METRIK & VISUALISASI

In [4]:
# =====================================================================
def calculate_accuracy_metrics(y_true, y_pred, y_train=None):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    errors, abs_errors = y_true - y_pred, np.abs(y_true - y_pred)
    mae, rmse = np.mean(abs_errors), np.sqrt(np.mean(errors**2))

    mask = y_true != 0
    mape = np.mean(abs_errors[mask] / np.abs(y_true[mask])) * 100 if np.any(mask) else np.nan

    denom = np.abs(y_true) + np.abs(y_pred)
    s_mask = denom != 0
    smape = np.mean(200 * abs_errors[s_mask] / denom[s_mask]) if np.any(s_mask) else np.nan

    mase = np.nan
    if y_train is not None and len(y_train) > 1:
        naive_mae = np.mean(np.abs(np.diff(y_train)))
        if naive_mae != 0: mase = mae / naive_mae

    return {'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape, 'sMAPE (%)': smape, 'MASE': mase}

def plot_eda(series, lags=20, title="Eksplorasi Data Deret Waktu (EDA)", save_dir=None):
    from statsmodels.tsa.seasonal import seasonal_decompose
    from statsmodels.tsa.stattools import adfuller, kpss
    from scipy.stats import boxcox, boxcox_llf

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    # --- 1. Uji Stasioneritas (Teks Report) ---
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series, autolag='AIC')
    kpss_stat, kpss_p, _, kpss_crit = kpss(series, regression="c", nlags="auto")

    report_text = []
    report_text.append("="*50)
    report_text.append("LAPORAN EKSPLORASI DATA DERET WAKTU (EDA)")
    report_text.append("="*50)
    report_text.append("\n[1] Uji Stasioneritas dalam Nilai Tengah:")
    report_text.append(f"  - ADF Test (H0: Tidak stasioner): p-value = {adf_p:.4f} -> {'Stasioner' if adf_p < 0.05 else 'Tidak Stasioner'}")
    report_text.append(f"  - KPSS Test (H0: Stasioner): p-value = {kpss_p:.4f} -> {'Tidak Stasioner' if kpss_p < 0.05 else 'Stasioner'}")

    # --- 2. Box-Cox Analysis ---
    try:
        # Pengecekan Box-Cox hanya bisa jika semua data positif
        if (series > 0).all():
            data_bc, lambda_fit, ci = boxcox(series, alpha=0.05)
            report_text.append("\n[2] Uji Stasioneritas dalam Ragam (Box-Cox Transformation):")
            report_text.append(f"  - Nilai Lambda Optimum: {lambda_fit:.4f}")
            report_text.append(f"  - 95% Confidence Interval: ({ci[0]:.4f}, {ci[1]:.4f})")

            is_lambda_one_in_ci = ci[0] <= 1 <= ci[1]
            report_text.append(f"  - Apakah Lambda=1 dalam CI? {'Ya (Tidak perlu transformasi)' if is_lambda_one_in_ci else 'Tidak (Perlu transformasi)'}")

            # Plot Box-Cox
            lambdas = np.linspace(-2, 3, 100)
            llf = [boxcox_llf(l, series) for l in lambdas]

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.plot(lambdas, llf, lw=2)
            ax.axvline(lambda_fit, color='r', linestyle='--', label=f'Optimal Lambda: {lambda_fit:.2f}')
            ax.axvline(1, color='green', linestyle='-.', label='Lambda = 1 (No Transform)')
            ax.axvspan(ci[0], ci[1], color='orange', alpha=0.3, label=f'95% CI: ({ci[0]:.2f}, {ci[1]:.2f})')

            title_text = f"Box-Cox Transformation Plot\nLambda=1 dalam CI? {'Ya' if is_lambda_one_in_ci else 'Tidak'}"
            ax.set(xlabel='Lambda', ylabel='Log-Likelihood Function', title=title_text)
            ax.legend()
            ax.grid(True)
            if save_dir:
                try: plt.savefig(os.path.join(save_dir, "01_BoxCox_Transformation.png"), bbox_inches='tight')
                except OSError: pass
            plt.close(fig)
        else:
            report_text.append("\n[2] Uji Stasioneritas dalam Ragam: Dilewati (Terdapat nilai <= 0)")
    except Exception as e:
        report_text.append(f"\n[2] Uji Stasioneritas dalam Ragam: Gagal ({e})")

    if save_dir:
        with open(os.path.join(save_dir, "EDA_Report.txt"), "w") as f:
            f.write("\n".join(report_text))

    # --- 3. Plot Time Series (Utama) ---
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(series.index, series.values, label='Observasi')
    ax.set(title=f"{title} - Time Series", xlabel="Waktu", ylabel="Nilai")
    ax.legend()
    if save_dir:
        try: plt.savefig(os.path.join(save_dir, "02_TimeSeries_Plot.png"), bbox_inches='tight')
        except OSError: pass
    plt.close(fig)

    # --- 4. Plot ACF & PACF ---
    fig, (ax_acf, ax_pacf) = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(series, lags=lags, ax=ax_acf, alpha=0.05)
    plot_pacf(series, lags=lags, ax=ax_pacf, alpha=0.05)
    ax_acf.set_title("Autocorrelation Function (ACF)")
    ax_pacf.set_title("Partial Autocorrelation (PACF)")
    plt.tight_layout()
    if save_dir:
        try: plt.savefig(os.path.join(save_dir, "03_ACF_PACF.png"), bbox_inches='tight')
        except OSError: pass
    plt.close(fig)

    # --- 5. Seasonal Decomposition ---
    period = 12 if len(series) >= 24 else (4 if len(series) >= 8 else 2)
    try:
        decomp = seasonal_decompose(series, model='additive', period=period)
        fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
        axes[0].plot(series.index, decomp.trend, color='orange')
        axes[0].set_title(f"Komponen Tren (Dekomposisi Aditif, period={period})")
        axes[1].plot(series.index, decomp.seasonal, color='green')
        axes[1].set_title("Komponen Musiman (Seasonality)")
        axes[2].plot(series.index, decomp.resid, color='red')
        axes[2].set_title("Komponen Sisaan (Residuals)")
        plt.tight_layout()
        if save_dir:
            try: plt.savefig(os.path.join(save_dir, "04_Seasonal_Decomposition.png"), bbox_inches='tight')
            except OSError: pass
        plt.close(fig)
    except Exception:
        pass

    # --- 6. Lag Plot ---
    fig, ax = plt.subplots(figsize=(6, 6))
    pd.plotting.lag_plot(series, ax=ax, c='purple', alpha=0.6)
    ax.set_title("Lag Plot (Y_t vs Y_{t-1})")
    if save_dir:
        try: plt.savefig(os.path.join(save_dir, "05_Lag_Plot.png"), bbox_inches='tight')
        except OSError: pass
    plt.close(fig)

In [5]:
# =====================================================================

In [6]:
# 2. PREPROCESSING & EACF HELPERS

In [7]:
# =====================================================================
class DynamicTimeSeriesPreprocessor:
    def __init__(self, handle_outliers=True, iqr_multiplier=1.5, auto_boxcox=False):
        self.handle_outliers, self.iqr_multiplier, self.auto_boxcox = handle_outliers, iqr_multiplier, auto_boxcox
        self.boxcox_lambda, self.shift_constant = None, 0.0

    def preprocess(self, series: pd.Series) -> pd.Series:
        s = series.iloc[:, 0].copy() if isinstance(series, pd.DataFrame) else series.copy()
        print("\n[ TAHAP 0: PREPROCESSING ]")

        if s.isna().sum() > 0:
            s = s.interpolate(method='linear').bfill().ffill()
            print(" -> Missing values terimputasi.")

        if self.handle_outliers:
            q1, q3 = s.quantile(0.25), s.quantile(0.75)
            bound = self.iqr_multiplier * (q3 - q1)
            s = s.clip(lower=q1 - bound, upper=q3 + bound)
            print(" -> Outliers di-cap berbasis IQR.")

        if self.auto_boxcox:
            self.shift_constant = abs(s.min()) + 1.0 if s.min() <= 0 else 0.0
            tx, lmbda = boxcox(s + self.shift_constant)
            self.boxcox_lambda = None if abs(lmbda - 1.0) < 0.1 else lmbda
            if self.boxcox_lambda:
                s = pd.Series(tx, index=s.index)
                print(f" -> Box-Cox diterapkan (Lambda={lmbda:.4f}).")

        return s

    def inverse_transform(self, vals):
        vals = np.asarray(vals, float)
        return inv_boxcox(vals, self.boxcox_lambda) - self.shift_constant if self.boxcox_lambda else vals

def calculate_eacf(series, ar_max=5, ma_max=5):
    z = np.asarray(series, float)
    z = z[~np.isnan(z)]
    n = len(z)
    z_c = z - np.mean(z)

    eacf_mat = np.zeros((ar_max + 1, ma_max + 1))
    res = {(0, j): z_c for j in range(ma_max + 1)}
    eacf_mat[0, :] = acf(z_c, nlags=ma_max + 1, fft=False)[1:ma_max + 2]

    for k in range(1, ar_max + 1):
        for j in range(ma_max + 1):
            Y = z_c[k:]
            X = np.column_stack([z_c[k-i : n-i] for i in range(1, k + 1)] + ([res[(k-1, j-1)][:-1][-(n-k):]] if j > 0 else []))
            phi = np.linalg.lstsq(X, Y, rcond=None)[0]
            res[(k, j)] = Y - X @ phi

            W = Y - X[:, :k] @ phi[:k]
            eacf_mat[k, j] = acf(W, nlags=j + 1, fft=False)[j + 1] if len(W) > j + 1 else np.nan

    bound = norm.ppf(0.975) / np.sqrt(np.maximum(n - np.arange(ar_max + 1)[:, None] - np.arange(ma_max + 1), 1))
    return pd.DataFrame(np.where(np.isnan(eacf_mat), '-', np.where(np.abs(eacf_mat) > bound, 'x', 'o')),
                        index=[f"AR {k}" for k in range(ar_max + 1)], columns=[f"MA {j}" for j in range(ma_max + 1)])

def extract_eacf_candidates(df_eacf):
    mat = (df_eacf.values == 'o')
    candidates = []
    max_p, max_q = mat.shape

    for p in range(max_p):
        for q in range(max_q):
            if mat[p, q]:
                # Triangle rule: for all k >= 0, row p+k from column q+k onwards must be 'o'
                is_triangle = True
                for k in range(max_p - p):
                    col_start = min(q + k, max_q - 1)
                    if not all(mat[p + k, col_start:]):
                        is_triangle = False
                        break
                if is_triangle:
                    candidates.append((p, q))

    if not candidates: return (0, 0), [(0,0)]

    candidates = sorted(candidates, key=lambda x: (x[0]+x[1], x[1]))
    return candidates[0], candidates[:min(5, len(candidates))]

In [8]:
# =====================================================================

In [9]:
# 3. ANALYZER UTAMA

In [10]:
# =====================================================================
class BoxJenkinsAnalyzer:
    def __init__(self, preprocessor=None, max_d=2, alpha=0.05, max_p=7, max_q=7, output_dir="output"):
        self.preprocessor = preprocessor or DynamicTimeSeriesPreprocessor()
        self.max_d, self.alpha = max_d, alpha
        self.max_p, self.max_q = max_p, max_q

        self.original_series = None
        self.clean_series = None
        self.d_estimated = 0

        self.best_orders = {}
        self.best_models = {}
        self.parameter_results = {}
        self.final_order = None
        self.final_model = None

        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)

    def _difference_until_stationary(self, s):
        def test(s):
            return adfuller(s)[1], kpss(s)[1]

        d, current_s = 0, s.copy()
        if self.max_d > 0:
            p_adf, p_kpss = test(current_s)
            while (p_adf > self.alpha or p_kpss < self.alpha) and d < self.max_d:
                d += 1
                current_s = current_s.diff().dropna()
                p_adf, p_kpss = test(current_s)
        return d, current_s

    def _estimate_adaptive_bounds(self, s):
        conf = 1.96 / np.sqrt(len(s))
        nlags = min(20, len(s) // 2 - 1)
        acf_v, pacf_v = acf(s, nlags=nlags, fft=False), pacf(s, nlags=nlags, method='yw')

        self.max_q = max(1, min(self.max_q, next((lag for lag in range(nlags, 0, -1) if abs(acf_v[lag]) > conf), 0)))
        self.max_p = max(1, min(self.max_p, next((lag for lag in range(nlags, 0, -1) if abs(pacf_v[lag]) > conf), 0)))
        print(f" -> Adaptive Bounds: max_p={self.max_p}, max_q={self.max_q}")

    def run_pipeline(self, series, show_eda=False):
        self.raw_series = series.iloc[:, 0].copy() if isinstance(series, pd.DataFrame) else series.copy()
        self.clean_series = self.preprocessor.preprocess(self.raw_series)

        print("\n[ TAHAP 1: KESTASIONERAN & IDENTIFIKASI ]")
        self.d_estimated, self.stationary_series = self._difference_until_stationary(self.clean_series)
        print(f" -> Diferensiasi Terpilih: d = {self.d_estimated}")
        self._estimate_adaptive_bounds(self.stationary_series)

        if show_eda:
            eda_dir = os.path.join(self.output_dir, "EDA")
            plot_eda(self.raw_series, title=f"EDA Data Asli (Sebelum Preprocessing)", save_dir=eda_dir)

        print("\n[ TAHAP 2: ESTIMASI PARAMETER (EACF) ]")
        df_eacf = calculate_eacf(self.stationary_series, self.max_p, self.max_q)
        print(df_eacf)
        _, candidates = extract_eacf_candidates(df_eacf)
        print(f" -> Kandidat: {candidates}")

        self._fit_candidates(candidates)
        return self

    def _fit_candidates(self, candidates):
        results, models = [], {}

        for p, q in candidates:
            order = (p, self.d_estimated, q)
            try:
                res = ARIMA(self.clean_series, order=order).fit()
                models[order] = res
                results.append({'Order': order, 'AIC': res.aic, 'BIC': res.bic, 'AICc': res.aicc, 'HQIC': res.hqic, 'LOGLIKE': res.llf})
            except Exception: pass

        df_res = pd.DataFrame(results)
        print("\n[ REKAPITULASI METRIK ]")
        print(df_res.to_string(index=False))

        for crit in ['AIC', 'BIC', 'AICc', 'HQIC']:
            self.best_orders[crit] = df_res.loc[df_res[crit].idxmin(), 'Order']
            self.best_models[crit] = models[self.best_orders[crit]]
        self.best_orders['LOGLIKE'] = df_res.loc[df_res['LOGLIKE'].idxmax(), 'Order']
        self.best_models['LOGLIKE'] = models[self.best_orders['LOGLIKE']]

    def estimate_parameters(self, criterion="AICc", save=True):
        """Menampilkan dan menyimpan estimasi parameter model terbaik.

        Statsmodels melakukan estimasi parameter ketika ARIMA(...).fit() dipanggil.
        Fungsi ini mengambil model terpilih, merangkum parameter, standard error,
        statistik z, p-value, dan interval kepercayaan 95%.
        """
        if not self.best_models:
            raise RuntimeError("Belum ada model yang diestimasi. Jalankan run_pipeline() terlebih dahulu.")

        if criterion not in self.best_models:
            raise ValueError(f"Kriteria '{criterion}' tidak tersedia. Pilih dari: {list(self.best_models.keys())}")

        order = self.best_orders[criterion]
        model = self.best_models[criterion]
        self.final_order = order
        self.final_model = model

        ci = model.conf_int(alpha=self.alpha)
        rows = []
        for i, name in enumerate(model.param_names):
            rows.append({
                "Parameter": name,
                "Estimate": model.params[i],
                "Std_Error": model.bse[i],
                "z_value": model.tvalues[i],
                "p_value": model.pvalues[i],
                "CI_2.5%": ci.iloc[i, 0] if hasattr(ci, "iloc") else ci[i, 0],
                "CI_97.5%": ci.iloc[i, 1] if hasattr(ci, "iloc") else ci[i, 1],
                "Signif_5%": "Ya" if model.pvalues[i] < self.alpha else "Tidak"
            })

        df_params = pd.DataFrame(rows)
        self.parameter_results[criterion] = df_params

        print("\\n" + "=" * 90)
        print(f"[ TAHAP 3A: ESTIMASI PARAMETER MODEL TERPILIH ({criterion}) ]")
        print("=" * 90)
        print(f"Model terpilih : ARIMA{order}")
        print("Metode estimasi: Maximum Likelihood (ML) melalui ARIMA.fit()")
        print("\nRingkasan estimasi parameter:")
        print(df_params.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
        print("\nPersamaan model (koefisien berdasarkan hasil estimasi):")
        print(model.summary())

        if save:
            crit_dir = os.path.join(self.output_dir, criterion)
            os.makedirs(crit_dir, exist_ok=True)
            csv_path = os.path.join(crit_dir, "Parameter_Estimates.csv")
            txt_path = os.path.join(crit_dir, "Parameter_Estimates.txt")
            try:
                df_params.to_csv(csv_path, index=False)
                with open(txt_path, "w", encoding="utf-8") as f:
                    f.write(f"Model terpilih: ARIMA{order}\\n")
                    f.write("Metode estimasi: Maximum Likelihood (ML) melalui ARIMA.fit()\\n\\n")
                    f.write(df_params.to_string(index=False))
                print(f"\\n -> Tabel parameter disimpan: {csv_path}")
                print(f" -> Ringkasan parameter disimpan: {txt_path}")
            except OSError as e:
                print(f"[!] Gagal menyimpan hasil estimasi parameter: {e}")

        return df_params

    def diagnostics(self):
        from statsmodels.stats.diagnostic import het_arch, het_breuschpagan
        import statsmodels.api as sm
        print("\n" + "="*80)
        print("[ TAHAP 3: DIAGNOSTIK SISAAN (KOMPREHENSIF) ]")
        print("="*80)

        any_passed = False
        # Deduplicate models that were selected by multiple criteria
        unique_models = {}
        for crit, order in self.best_orders.items():
            if order not in unique_models:
                unique_models[order] = {'criteria': [crit], 'model': self.best_models[crit]}
            else:
                unique_models[order]['criteria'].append(crit)

        for order, data in unique_models.items():
            crits = ", ".join(data['criteria'])
            model = data['model']
            resids = model.resid.dropna()

            p, d, q = order
            model_df = p + q
            lb_lags = max(10, model_df + 5)
            arch_lags = min(10, len(resids)//3)

            # 1. Ljung-Box
            lb_res = acorr_ljungbox(resids, lags=[lb_lags], model_df=model_df, return_df=True)
            lb_stat, lb_p = lb_res['lb_stat'].iloc[0], lb_res['lb_pvalue'].iloc[0]

            # 2. Normality
            sw_stat, sw_p = shapiro(resids)
            jb_stat, jb_p, skew, kurt = jarque_bera(resids)

            # 3. Heteroscedasticity (ARCH-LM & Breusch-Pagan)
            arch_res = het_arch(resids, nlags=arch_lags)
            arch_stat, arch_p = arch_res[0], arch_res[1]

            # Breusch-Pagan (Test if variance changes linearly over time)
            bp_exog = sm.add_constant(np.arange(len(resids)))
            bp_stat, bp_p, _, _ = het_breuschpagan(resids, bp_exog)

            # 4. Parameters
            insig_params = model.pvalues[model.pvalues > self.alpha]

            def conclude(pval):
                return "TOLAK H0" if pval < self.alpha else "GAGAL TOLAK H0"

            print(f"\n>> MODEL: ARIMA{order} (Dipilih oleh: {crits})")
            print("-" * 80)

            print("1. UJI AUTOKORELASI (Ljung-Box Test)")
            print("   [H0: Residual saling bebas | H1: Terdapat autokorelasi]")
            print(f"   - Setup      : Lags={lb_lags}, Model_DF={model_df}")
            print(f"   - Statistik  : Q = {lb_stat:.4f}, P-Value = {lb_p:.4f}")
            print(f"   - Kesimpulan : {conclude(lb_p)} -> {'Terdapat sisaan autokorelasi (Gagal)' if lb_p < self.alpha else 'Residual saling bebas (Lolos)'}")

            print("\n2. UJI NORMALITAS (Shapiro-Wilk & Jarque-Bera)")
            print("   [H0: Residual berdistribusi normal | H1: Residual tidak berdistribusi normal]")
            print(f"   - Shapiro-W  : W = {sw_stat:.4f}, P-Value = {sw_p:.4f} -> {conclude(sw_p)}")
            print(f"   - Jarque-Bera: JB = {jb_stat:.4f}, P-Value = {jb_p:.4f} -> {conclude(jb_p)}")
            print(f"   - Momen      : Skewness = {skew:.4f}, Kurtosis = {kurt:.4f}")

            print("\n3. UJI HOMOSKEDASTISITAS (ARCH-LM & Breusch-Pagan)")
            print("   [H0: Varians residual konstan | H1: Varians residual tidak konstan (Heteroskedastisitas)]")
            print(f"   - ARCH-LM    (Lags={arch_lags}): LM = {arch_stat:.4f}, P-Value = {arch_p:.4f} -> {conclude(arch_p)}")
            print(f"   - Breusch-Pagan (Trend): LM = {bp_stat:.4f}, P-Value = {bp_p:.4f} -> {conclude(bp_p)}")
            if arch_p < self.alpha or bp_p < self.alpha:
                print("   - Kesimpulan : TOLAK H0 -> Varians tidak konstan (Gagal)")
            else:
                print("   - Kesimpulan : GAGAL TOLAK H0 -> Homoskedastis (Lolos)")

            print("\n4. SIGNIFIKANSI PARAMETER MODEL (Uji-t)")
            print("   [H0: Parameter = 0 (Tidak Signifikan) | H1: Parameter != 0 (Signifikan)]")
            if len(insig_params) > 0:
                print(f"   - Parameter Tidak Signifikan (P-Val > {self.alpha}):")
                for name, pval in insig_params.items():
                    print(f"     * {name} (P-Val = {pval:.4f})")
            else:
                print("   - Seluruh parameter signifikan (Lolos).")
            print("-" * 80)

            # --- Diagnostic Visualizations Split per Element ---

            # 1. Autokorelasi
            fig, ax = plt.subplots(figsize=(8, 5))
            plot_acf(resids, lags=lb_lags, ax=ax, alpha=self.alpha)
            lb_text = f"Ljung-Box Q: {lb_stat:.2f} | P-Value: {lb_p:.4f}\nKesimpulan: {'Terdapat Autokorelasi (Gagal)' if lb_p < self.alpha else 'Residual Saling Bebas (Lolos)'}"
            ax.set_title(f"Autokorelasi Sisaan (ARIMA{order})\n{lb_text}")
            plt.tight_layout()
            for crit in data['criteria']:
                crit_dir = os.path.join(self.output_dir, crit)
                os.makedirs(crit_dir, exist_ok=True)
                try: plt.savefig(os.path.join(crit_dir, f"Diag_01_Autokorelasi_ARIMA_{order[0]}_{order[1]}_{order[2]}.png"), bbox_inches='tight')
                except OSError: pass
            plt.close(fig)

            # 2. Normalitas
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            sns.histplot(resids, kde=True, ax=axes[0])
            qqplot(resids, line='q', ax=axes[1])
            norm_text = f"Shapiro-W (p={sw_p:.4f}) | Jarque-Bera (p={jb_p:.4f})"
            fig.suptitle(f"Normalitas Sisaan (ARIMA{order})\n{norm_text}", y=1.05)
            plt.tight_layout()
            for crit in data['criteria']:
                crit_dir = os.path.join(self.output_dir, crit)
                os.makedirs(crit_dir, exist_ok=True)
                try: plt.savefig(os.path.join(crit_dir, f"Diag_02_Normalitas_ARIMA_{order[0]}_{order[1]}_{order[2]}.png"), bbox_inches='tight')
                except OSError: pass
            plt.close(fig)

            # 3. Homoskedastisitas
            fig, ax = plt.subplots(figsize=(8, 5))
            plot_acf(resids**2, lags=arch_lags, ax=ax, alpha=self.alpha)
            homo_text = f"ARCH-LM (p={arch_p:.4f}) | Breusch-Pagan (p={bp_p:.4f})\nKesimpulan: {'Heteroskedastis (Gagal)' if (arch_p < self.alpha or bp_p < self.alpha) else 'Homoskedastis (Lolos)'}"
            ax.set_title(f"Homoskedastisitas (ACF Sisaan Kuadrat) (ARIMA{order})\n{homo_text}")
            plt.tight_layout()
            for crit in data['criteria']:
                crit_dir = os.path.join(self.output_dir, crit)
                os.makedirs(crit_dir, exist_ok=True)
                try: plt.savefig(os.path.join(crit_dir, f"Diag_03_Homoskedastisitas_ARIMA_{order[0]}_{order[1]}_{order[2]}.png"), bbox_inches='tight')
                except OSError: pass
            plt.close(fig)

            # 4. Signifikansi Parameter (Teks pada Gambar)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.axis('off')
            if len(insig_params) > 0:
                param_text = "Parameter Tidak Signifikan (p > 0.05):\n\n" + "\n".join([f"• {k}: p-value = {v:.4f}" for k, v in insig_params.items()])
            else:
                param_text = "Seluruh parameter signifikan (Lolos)."
            ax.text(0.5, 0.5, f"Signifikansi Parameter (ARIMA{order})\n\n{param_text}", ha='center', va='center', fontsize=12, wrap=True)
            for crit in data['criteria']:
                crit_dir = os.path.join(self.output_dir, crit)
                os.makedirs(crit_dir, exist_ok=True)
                try: plt.savefig(os.path.join(crit_dir, f"Diag_04_Signifikansi_ARIMA_{order[0]}_{order[1]}_{order[2]}.png"), bbox_inches='tight')
                except OSError: pass
            plt.close(fig)

            # Determine if this model passed critical diagnostics
            passed_autocorr = (lb_p >= self.alpha)
            passed_homosced = (arch_p >= self.alpha and bp_p >= self.alpha)
            passed_params = (len(insig_params) == 0)
            if passed_autocorr and passed_homosced and passed_params:
                any_passed = True

        return any_passed

    def evaluate(self, test_series, plot=True):
        print("\n[ TAHAP 4: EVALUASI TESTING OUT-OF-SAMPLE ]")
        test_series = test_series.iloc[:, 0] if isinstance(test_series, pd.DataFrame) else test_series
        preds, results = {}, []

        valid_preds = {}
        for crit, model in self.best_models.items():
            y_pred = self.preprocessor.inverse_transform(model.get_forecast(steps=len(test_series)).predicted_mean)

            # Jika hasil forecast menghasilkan NaN (model meledak/tidak stabil), skip model ini
            if np.isnan(y_pred).any():
                continue

            valid_preds[crit] = pd.Series(y_pred, index=test_series.index)
            metrics = calculate_accuracy_metrics(test_series, y_pred, self.raw_series)
            results.append({'Kriteria': crit, 'Model': f"ARIMA{self.best_orders[crit]}", **metrics})

        if not results:
            print("   [!] Semua model menghasilkan forecast yang tidak valid (NaN).")
            return

        print(pd.DataFrame(results).to_string(index=False))

        if plot:
            for crit, y_pred in valid_preds.items():
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.plot(self.raw_series.index, self.raw_series, label='Data Latih')
                ax.plot(test_series.index, test_series, label='Data Aktual (Uji)')
                ax.plot(test_series.index, y_pred, label=f'Forecast ({crit})', color='red', linestyle='--')
                ax.set_title(f"Forecast vs Aktual - ARIMA{self.best_orders[crit]} ({crit})")
                ax.legend()

                crit_dir = os.path.join(self.output_dir, crit)
                os.makedirs(crit_dir, exist_ok=True)
                save_path = os.path.join(crit_dir, "Forecast_Out_Of_Sample.png")
                try:
                    plt.savefig(save_path, bbox_inches='tight')
                except OSError as e:
                    print(f"   [!] Gagal menyimpan {save_path} (File mungkin sedang dibuka): {e}")
                plt.close(fig)

    def evaluate_cv(self, initial_ratio=0.8, horizon=1):
        print("\n[ TAHAP 4.5: ROLLING CROSS-VALIDATION ]")
        y = self.clean_series.values
        n_init = int(len(y) * initial_ratio)
        results = []

        for crit, order in self.best_orders.items():
            fold_metrics = []
            for t in range(n_init, len(y) - horizon + 1):
                try:
                    res = ARIMA(y[:t], order=order).fit()
                    pred = self.preprocessor.inverse_transform(res.get_forecast(steps=horizon).predicted_mean)

                    if np.isnan(pred).any():
                        continue

                    truth = self.preprocessor.inverse_transform(y[t:t+horizon])

                    fold_metrics.append(calculate_accuracy_metrics(truth, pred, y[:t]))
                except Exception: pass

            if fold_metrics:
                results.append({'Kriteria': crit, 'Model': f"ARIMA{order}", 'Folds': len(fold_metrics),
                                'Avg MAE': np.mean([m['MAE'] for m in fold_metrics]),
                                'Avg RMSE': np.mean([m['RMSE'] for m in fold_metrics]),
                                'Avg MAPE (%)': np.nanmean([m['MAPE (%)'] for m in fold_metrics])})

        print(pd.DataFrame(results).to_string(index=False))

    def forecast(self, steps=5, plot=True):
        print("\n[ TAHAP 5: FUTURE FORECASTING ]")
        res = {}

        for crit, model in self.best_models.items():
            y_pred = self.preprocessor.inverse_transform(model.get_forecast(steps=steps).predicted_mean)

            if np.isnan(y_pred).any():
                continue

            res[f"{crit} ARIMA{self.best_orders[crit]}"] = y_pred

            if plot:
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.plot(self.raw_series.index, self.raw_series, label='History')
                idx = range(len(self.raw_series), len(self.raw_series) + steps)
                ax.plot(idx, y_pred, label='Forecast', color='orange', linestyle='--')
                ax.set_title(f"Future Forecast - ARIMA{self.best_orders[crit]} ({crit})")
                ax.legend()

                crit_dir = os.path.join(self.output_dir, crit)
                os.makedirs(crit_dir, exist_ok=True)
                save_path = os.path.join(crit_dir, "Future_Forecast.png")
                try:
                    plt.savefig(save_path, bbox_inches='tight')
                except OSError as e:
                    print(f"   [!] Gagal menyimpan {save_path} (File mungkin sedang dibuka): {e}")
                plt.close(fig)

        df = pd.DataFrame(res)
        print(df)
        return df

In [11]:
# =====================================================================

In [12]:
# 4. EKSEKUSI PIPELINE

In [13]:
# =====================================================================
if __name__ == "__main__":

    print("\n\n>>> 2. EKSEKUSI DATA PENUMPANG SOEKARNO-HATTA <<<")
    url = "https://raw.githubusercontent.com/lailynissa/tsapython/main/penumpang%20soeta.csv"
    df_soetta = pd.read_csv(url)
    kolom_waktu = df_soetta.columns[0]
    df_soetta[kolom_waktu] = pd.to_datetime(df_soetta[kolom_waktu])
    df_soetta = df_soetta.sort_values(by=kolom_waktu).reset_index(drop=True)

    n_train = int(len(df_soetta) * 0.95)
    train_data = df_soetta['Total'].iloc[:n_train]
    test_data = df_soetta['Total'].iloc[n_train:]

    max_retries = 3
    base_p, base_q = 7, 7

    for attempt in range(max_retries):
        current_p = base_p + (attempt * 7)
        current_q = base_q + (attempt * 7)

        print(f"\n{'#'*80}")
        print(f"--- ATTEMPT {attempt+1} (max_p={current_p}, max_q={current_q}) ---")
        print(f"{'#'*80}")

        analyzer_soetta = BoxJenkinsAnalyzer(
            DynamicTimeSeriesPreprocessor(auto_boxcox=True),
            max_d=np.ceil(np.sqrt(n_train)),
            max_p=current_p,
            max_q=current_q
        )

        analyzer_soetta.run_pipeline(train_data, show_eda=(attempt == 0))
        passed = analyzer_soetta.diagnostics()

        if passed:
            print("\n[+] Ditemukan model yang lolos uji diagnostik kritikal (Autokorelasi, Homoskedastisitas, Signifikansi Parameter)!")
            break
        else:
            print(f"\n[-] Tidak ada model yang lolos semua uji diagnostik pada attempt {attempt+1}.")
            if attempt < max_retries - 1:
                print(" -> Mengekspansi ruang pencarian (max_p, max_q)...")
            else:
                print(" -> Batas percobaan maksimal tercapai. Melanjutkan dengan model terbaik yang ada.")

    # Estimasi dan tampilkan parameter model terbaik (berdasarkan AICc)
    analyzer_soetta.estimate_parameters(criterion="AICc", save=True)

    analyzer_soetta.evaluate(test_data, plot=True)
    analyzer_soetta.evaluate_cv()
    analyzer_soetta.forecast(steps=6)



>>> 2. EKSEKUSI DATA PENUMPANG SOEKARNO-HATTA <<<

################################################################################
--- ATTEMPT 1 (max_p=7, max_q=7) ---
################################################################################

[ TAHAP 0: PREPROCESSING ]
 -> Outliers di-cap berbasis IQR.
 -> Box-Cox diterapkan (Lambda=2.2415).

[ TAHAP 1: KESTASIONERAN & IDENTIFIKASI ]
 -> Diferensiasi Terpilih: d = 2
 -> Adaptive Bounds: max_p=7, max_q=7

[ TAHAP 2: ESTIMASI PARAMETER (EACF) ]
     MA 0 MA 1 MA 2 MA 3 MA 4 MA 5 MA 6 MA 7
AR 0    x    o    o    o    o    x    x    o
AR 1    x    o    o    o    o    o    o    o
AR 2    o    o    o    o    o    x    x    o
AR 3    o    o    o    o    o    x    x    o
AR 4    o    o    o    o    o    x    x    o
AR 5    o    o    o    o    o    o    o    o
AR 6    o    o    o    o    o    o    o    o
AR 7    o    o    o    o    o    o    o    o
 -> Kandidat: [(5, 0), (6, 0), (5, 1), (7, 0), (6, 1)]

[ REKAPITULASI METRIK ]
    Ord